# VeriForgot — Notebook 01: Train Baseline ResNet-18

Trains the baseline ResNet-18 on CIFAR-10 (all 50,000 samples).
Saves: `model_original.pth`, `forget_indices.pkl`, `retain_indices.pkl`

**Run on Kaggle with GPU accelerator enabled.**

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as transforms
import torchvision.models as models
import numpy as np, pickle

SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── Transforms ────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])

full_train = torchvision.datasets.CIFAR10('./data', True,  download=True, transform=transform_train)
testset    = torchvision.datasets.CIFAR10('./data', False, download=True, transform=transform_test)
train_loader = torch.utils.data.DataLoader(full_train, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = torch.utils.data.DataLoader(testset,    batch_size=256, shuffle=False, num_workers=2)

In [ ]:
# ── Build forget/retain split ─────────────────────────
np.random.seed(SEED)
forget_candidates = [i for i,(_, l) in enumerate(full_train) if l in [0,1]]
np.random.shuffle(forget_candidates)
forget_indices = forget_candidates[:500]
retain_indices = [i for i in range(len(full_train)) if i not in set(forget_indices)]
print(f'D_forget: {len(forget_indices)} | D_retain: {len(retain_indices)}')
with open('forget_indices.pkl','wb') as f: pickle.dump(forget_indices, f)
with open('retain_indices.pkl','wb') as f: pickle.dump(retain_indices, f)
print('Indices saved.')

In [ ]:
# ── Train baseline model ──────────────────────────────
model = models.resnet18(weights=None)
model.fc = nn.Linear(512, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

for epoch in range(30):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        criterion(model(x), y).backward()
        optimizer.step()
    scheduler.step()
    if (epoch+1) % 5 == 0:
        model.eval(); correct = total = 0
        with torch.no_grad():
            for x,y in test_loader:
                x,y=x.to(device),y.to(device)
                _,p=model(x).max(1)
                correct+=p.eq(y).sum().item(); total+=y.size(0)
        print(f'Epoch {epoch+1}/30 | TestAcc: {100.*correct/total:.2f}%')

torch.save(model.state_dict(), 'model_original.pth')
print('Baseline model saved: model_original.pth')